## Sample Weights

This notebook will cover exercise answer.

* Exercise 4.5

As we go along, there will be some explanations.

More importantly, this method can be applied not just within mean-reversion strategy but also other strategies as well. 

Most of the functions below can be found under research/Sampling.

Contact: boyboi86@gmail.com

In [ ]:
import numpy as np
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

%matplotlib inline

**Note**

Instead of trend strategy, we will continue to use mean-reversion strategy.

This part will deviate slightly from the exercise, but I assure you the outcome is somewhat similiar.

However using trend strategy, the idea which Dr Marco Lopez wish to express would be obvious since meta-labels would suffer from imbalance sample.

In [ ]:
dollar = pd.read_csv('../sample-data/dollar_bars.txt', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])

In [ ]:
dollar = rs.bband_as_side(data = dollar, 
                          window = 50, 
                          width = 0.001)

dollar['volatility'] = rs.vol(dollar['close'], span0 = 50)

# volatility is a return; cs_filter diffs are price points, so scale by price level
events = rs.cs_filter(dollar['close'], 
                    limit = dollar['volatility'].mean() * dollar['close'].mean())

vb = rs.vert_barrier(data = dollar['close'], 
                 events = events, 
                 period = 'days', 
                 freq = 1)

tb = rs.tri_barrier(data = dollar['close'], 
                    events = events, 
                    trgt = dollar['volatility'] * 5, 
                    min_req = 0.002, 
                    num_threads = 3, 
                    ptSl = [0,2],
                    t1 = vb, 
                    side = dollar['side'])

mlabel = rs.meta_label(data = dollar['close'], 
                       events = tb, 
                       drop = False) # when you have a side binary, you won't have rare labels usually

In [ ]:
mlabel['bin'].value_counts()

In [ ]:
# original data matrix report w/o random forest

rs.report_matrix(actual_data = mlabel, 
                 prediction_data = None, 
                 ROC = None)

In [ ]:
# Exercise 4.5a
# for convienience sake, we will just go for crossing average & volatility as features

X = dollar.drop(['open', 'high', 'low', 'close','cum_vol', 'cum_dollar', 'cum_ticks'], axis = 1)
X.dropna(inplace = True) # we lost quite abit of data

X = X.reindex(mlabel.index)
y = mlabel['bin']

n_estimators, max_depth, c_random_state = 500, 7, 42

# Random Forest Model
rf = RandomForestClassifier(max_depth = max_depth, 
                            n_estimators = n_estimators,
                            criterion = 'entropy',
                            #bootstrap=True,
                            #max_samples = av_uniqueness_by_coevent['tW'].mean()
                            oob_score = False,
                            class_weight = None, # We did not use class weight
                            random_state = c_random_state)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

rf.fit(X_train, y_train.values.ravel())

y_prob_test = rf.predict_proba(X_test)[:, 1] #here we are only interested in True positive
y_pred_test = rf.predict(X_test)

# We go for test data straight
rs.report_matrix(actual_data = y_test, 
                 prediction_data = y_pred_test, 
                 ROC = y_prob_test)

In [ ]:
rf_cls_wght_bln = RandomForestClassifier(max_depth = max_depth, 
                                    n_estimators = n_estimators,
                                    criterion = 'entropy',
                                    #bootstrap=True,
                                    #max_samples = av_uniqueness_by_coevent['tW'].mean()
                                    oob_score = False,
                                    class_weight = 'balanced', # We try balance weight only
                                    random_state = c_random_state)

rf_cls_wght_bln.fit(X_train, y_train.values.ravel())

y_prob_test0 = rf_cls_wght_bln.predict_proba(X_test)[:, 1] #here we are only interested in True positive
y_pred_test0 = rf_cls_wght_bln.predict(X_test)

# We go for test data straight
rs.report_matrix(actual_data = y_test, 
                 prediction_data = y_pred_test0, 
                 ROC = y_prob_test0)

In [ ]:
rf_cls_wght_sub_bln = RandomForestClassifier(max_depth = max_depth, 
                                    n_estimators = n_estimators,
                                    criterion = 'entropy',
                                    #bootstrap=True,
                                    #max_samples = av_uniqueness_by_coevent['tW'].mean()
                                    oob_score = False,
                                    class_weight = 'balanced_subsample', # We try balance weight only
                                    random_state = c_random_state)

rf_cls_wght_sub_bln.fit(X_train, y_train.values.ravel())

y_prob_test1 = rf_cls_wght_sub_bln.predict_proba(X_test)[:, 1] #here we are only interested in True positive
y_pred_test1 = rf_cls_wght_sub_bln.predict(X_test)

# We go for test data straight
rs.report_matrix(actual_data = y_test, 
                 prediction_data = y_pred_test1, 
                 ROC = y_prob_test1)

### Conclusion:

#### Accuracy

Class_weight = 'None'.
The Accuracy was about 0.5221075902726603.

Class_weight = 'balance'.
The Accuracy was about 0.5289240972733972.

Class_weight = 'balance_subsample'.
The Accuracy was about 0.5309506263817244 (Highest improvement).

#### Confusion Matrix

The power of a hypothesis test is the probability of making the correct decision if the alternative hypothesis is true, hence to reject a null hypothesis if found to be false.

In our case, null hypothesis is FP (Not True until proven, easily rejected). By focusing on TP, we will increase our ML model's accuracy.

The distribution Postive (TP and FP) experience an increase, when class_weight was set to 'balance' and 'balance_subsample'.

Again, with 'balance_subsample' experience higher allocation to Positive distributions within condusion matrix, which is reflected in their respective accuracy score.

Comparing class weight = 'None' and class_weight = 'balance_subsample', True Positive (TP) experienced a jump from 1937 to 2004. In short, when we set class_weight, we are asking ML model to focus on True region specifically TP.

This is correct and especially true when we have a high imbalance dataset with TP being a minority, this is also the reason why Dr Marco Lopez wanted to demostrate with an imbalance dataset with 2/3 of negatives meta-labels.

For more details refer to Advances in Financial Machine Learning, page 71 - 72, section 4.8.